<!--

    Gaia Data Processing and Analysis Consortium (DPAC) 
    Co-ordination Unit 9 Work Package 930
    
    (c) 2005-2025 Gaia DPAC
    
    This program is free software: you can redistribute it and/or modify
    it under the terms of the GNU General Public License as published by
    the Free Software Foundation, either version 3 of the License, or
    (at your option) any later version.

    This program is distributed in the hope that it will be useful,
    but WITHOUT ANY WARRANTY; without even the implied warranty of
    MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
    GNU General Public License for more details.

    You should have received a copy of the GNU General Public License
    along with this program.  If not, see <https://www.gnu.org/licenses/>.
    -->

This Gaia science exploitation example is built on [Apache Spark](https://spark.apache.org). The primary user interface uses web-based Jupyter notebooks, employing Python code and a suite of third-party packages and libraries for analysis, plotting, etc. A working knowledge of Gaia data releases to-date, Python and Structured Query Language is assumed.


The best way to familiarise yourself with the PySpark API on the platform is to examine the example notebooks that come bundled with it. These are:

1. Start here: platform familiarisation (this notebook);
1. Data holdings: a description of the data hosted on the platform;
1. Source counts over the sky: a simple example of querying and plotting;
1. Mean proper motions over the sky: another simple example;
1. Working with Gaia XP spectra: demonstrates trawling the large XP spectral dataset, and plotting of the spectra;
1. Analyses involving cross-match surveys: using Gaia data in conjunction with optical and infrared survey data;
1. Good astrometric solutions via ML Random Forest classifier: a more complex workflow demonstrating use of the Spark ML library in supervised machine learning;
1. Tips and tricks: some further usage tips and technical aspects of the platform including interactions and plotting.

You can find example notebooks in `demo/` in your home directory. Note that you will need to copy these (`right-click` > `copy`) elsewhere in your filespace to edit or run them. 

Newcomers are advised to work through at least the first four notebooks before attempting to use the system.

This and all the other example notebooks itemised above are Jupyter Notebooks. Text, Python code, and plots can be freely mixed in the notebook in paragraphs, or cells. For example, to see the [markdown code](https://jupyter-notebook.readthedocs.io/en/stable/examples/Notebook/Working%20With%20Markdown%20Cells.html) for this cell simply double click anywhere; execute the cell with `Ctrl` + (`Alt` + )`Shift` or the "play" button in the panel above to hide the code leaving only the output.  

To save a notebook locally, select `File` > `Download` in the menu at the top, or right-click the notebook in the file viewer on the left and select "Download". This will create a local file with extension ".ipynb". To import a previously saved note (or indeed one provisioned externally) click on the upload icon in the top left of the UI and select a file from your local file system.

Please note that:
* You are strongly advised to save locally all your notebook edits at frequent intervals (see above). Preservation of the notebook content of the platform can be managed on a best-efforts basis only. While we will endeavour to restore fully in the event of loss of the deployed system or Cloud services on which it is built, we can make no guarantee that your work will be backed up and will be recovered.
* Important etiquette: Astroflow is a resource shared between users. Please be considerate to fellow users by logging out of the web interface when you are not using it. In particular please do not leave notebooks open and holding cached data when you are not using them - further examples of best practice are given below.  


The platform environment employs [Spark (version 3.5.2)](https://spark.apache.org/docs/3.5.2/) for distributed processing and scalability to high-volume data releases. For convenience a high-level application programming interface (API) is available through Python to access and analyse large data sets. The primary distributed data container is known as a ["data frame"](https://spark.apache.org/docs/3.5.2/sql-programming-guide.html), a richly structured object encapsulating data, descriptions and methods that is _not_ limited to simple tables with primitive column types in contrast to systems built on relational data base technology (RDBMS). For ease of use the [API features Structured Query Language](https://spark.apache.org/docs/3.5.2/sql-ref.html) as a familiar way to create and manipulate data objects as the initial steps in work flows. Note that this is not Astronomy Data Query Language (ADQL): astronomy-specific aspects, in particular geometric functions for spatial queries, are not available in this flavour of SQL. However PySpark SQL offers a much greater degree of end-user programmability than ADQL interfaces built in front of RDBMS, for example User Defined Functions (UDFs) that can be integrated into SQL statements. The fact that SQL features prominently in the API should not be taken to mean that this platform aims to replicate the functionality of existing relational systems. Obviously, usage scenarios best served by such systems should be targetted there. This platform is designed to facilitate scale-out usage scenarios, e.g. large-scale statistical analyses, that cannot be achieved through relational systems built on TAP/ADQL. (In fact on this platform you can mix-and-match both by calling out to any IVOA TAP/ADQL service as part of a larger work flow.)

When deciding to use PySpark with Gaia data is is necessary to import a script to initialise the resources and load the database shcema. This can be done at the head of each notebook as follows:

    # Import the spark config
    from astroflow_spark_gaia import spark
        
... see the next code cell. The notebook interpreter can be set for individual cells as is done here and in the next by selecting from the drop-down menu at the cell settings above (which will currently be either "Code" or "Markdown")


In [4]:
# Import the spark config
from astroflow_spark_gaia import spark

# echo details of the available data resources
for line in spark.catalog.listTables(): print (line)

The operating model of Spark as actioned via a workflow expressed in a notebook will be unfamiliar to many. Users should be aware of the following basic facts from the outset:

__Lazy Evaluation:__ Spark employs "lazy evaluation" of jobs: nothing happens until the last possible point in the workflow. For example, a notebook cell may contain a data frame defined by a query 

    df = spark.sql("select * from gaia_source")

but as far as Spark is concerned this is simply a definiton, or "transformation", of a data resource. If nothing is done with the data frame defined by that query in the cell, then the cell will appear to run instantaneously. Only if an "action" is made on the data frame subsequently will physical execution take place. This means, for example, that if a cell contains a run-time error this may not show up until a later cell is executed with the exception trace explaining the error appearing in that later cell's output (as opposed to any output associated with the cell containing the source of the error). Note also that the statement above, on it's own, will not result in the entire billon+ row, ~100 column, tera-byte scale data set being loaded into memory. Whatever is expressed in subsequent transformations (e.g. SQL "where" clauses) and actions (e.g. aggregations) will define "filter push-down" optimisations that result in only those data needed to action the process being read into memory and processed.  

__Caching:__ Actions expressed in a cell may need to be re-executed if needed as part of the execution plan for actions in a subsequent cell. This may result in expensive operations being repeatedly executed when developing notebook workflows and working on cells in isolation. You can tell Spark to cache a data frame explicitly to avoid this and speed-up significantly subsequent cell operations

    df = spark.sql("select column_1, column_2, ... from gaia_source where ...").cache()
    
but obviously this should only be done within reason and with due regard to the amount of local memory/disk required by the worker nodes: make sure you only cache what you really need and _in particular always free up cached resources at the end of your notebook session:_


    sqlContext.clearCache()

Some of the example notebooks provided use caching to expedite the execution of cells that re-use data frames.

__Indexing:__ The data hosted on this platform are arranged by gaia_source.source_id as the primary unique identifier in Gaia Data Releases. This key should be employed when joining distinct data sets, e.g. pre-computed best neighbours from external survey catalogues - see the "Data holdings" notebook. It can also be used to pull out specific records quickly:

    spark.sql("select * from gaia_source where source_id = ...").show()
   
Otherwise, the general RDBMS paradigm employing SQL indexes to optimise execution plans of queries predicated on certain columns has no equivalent here. It is worth noting however that the combination of "filter push-down" and the underlying storage format ([Parquet](https://parquet.apache.org/docs/)) result in only those data that are needed being read from disk on this platform. This is in contrast to RDBMS where columns are generally read from pages within disk files regardless of need unless covered by an index employed as part of an optimised execution plan. This is one of the ways in which Spark achieves much faster performance in "big data" analysis.

__Distributed processing model:__ Fast performance in handling very large data sets is achieved via cluster parallelism with actions split up and optimised for execution on concurrent workers. This is achievable only if the user's processing works with the Spark data frame which in turn encapsulates a Spark Resilient Distributed Dataset (RDD). Note that issuing a `sparkDataFrame.collect()`, or `sparkDataFrame.toPandas()` for example, retrieves the distributed components of the Spark data frame to a single driver process and subsequent actions will be no longer distributed over the cluster of workers. Care should be taken in the final stages of work flows to avoid Out Of Memory exceptions by collecting too large a results set to the driver process.



**Requesting more resources:** If you find that you have insufficient memory or compute power, you can request a Spark session with more resources.

For example, to request 10 pods, each with 20 CPUs and 30 GB of memory, use:
```
    from spark_setup import SparkSetup
    from setup_data_gaia_dr3 import SetupDataGaiaDR3
    spark_setup = SparkSetup(pods=10, cpu=20, mem=30, data_setup=SetupDataGaiaDR3)
    spark = spark_setup.get_spark_session()
```

While we ask that you are considerate of other users by not requesting excessive resources, we do encourage users to request enough resources to complete their work efficiently. Idle resources will be released after a few minutes, and some limits are imposed automatically by the system. 

In addition to the example notebooks provided here and the linked resources therein see the following for further information:

* [PySpark documentation](https://spark.apache.org/docs/latest/api/python/index.html)
* [PySpark SQL cheat sheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf)

